<a href="https://colab.research.google.com/github/smilingSG/CSCI218-Spam-Email-Detection-Project/blob/main/CSCI218_Group48_SpamDetectionEmail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
# 1)Load dataset
# Upload the file into colab directly
df = pd.read_csv("spam_ham_dataset.csv")

In [ ]:
#2)Quick sanity checks
#Drop the useless index column if it exists
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Shape:", df.shape)
print(df["label"].value_counts())

X = df["text"].astype(str)
y = df["label"]  # "ham" / "spam"

In [ ]:
#3)Train-test split (stratify keeps spam/ham ratio consistent)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
#4)Shared TF-IDF settings (unigrams only)
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 1),     # unigrams only
    min_df=2,               # ignore very rare words (reduces noise)
    max_df=0.95             # ignore overly common words
)

In [ ]:
#5)Baseline: Multinomial Naive Bayes
nb_model = Pipeline([
    ("tfidf", tfidf),
    ("clf", MultinomialNB())
])

nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

print("\n=== Multinomial Naive Bayes (Baseline) ===")
print("Accuracy:", accuracy_score(y_test, nb_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, nb_pred, labels=["ham", "spam"]))
print(classification_report(y_test, nb_pred, digits=4))

In [ ]:
#6)Comparison: Logistic Regression
#Class imbalance exists (more ham than spam), so class_weight="balanced" helps fairness
lr_model = Pipeline([
    ("tfidf", tfidf),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print("\n=== Logistic Regression (Comparison) ===")
print("Accuracy:", accuracy_score(y_test, lr_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, lr_pred, labels=["ham", "spam"]))
print(classification_report(y_test, lr_pred, digits=4))
